# Xem accuracy_log — tự lấy epoch accuracy CAO NHẤT rồi in full mẫu

Mỗi dòng JSONL = 1 lần eval (1 epoch): `accuracy`, `samples` (tất cả, gọn: idx/pred/gold/correct), `detail_samples` (6 mẫu full: gold vs prediction kèm `premises_used` + `reasoning_steps` + `explanation`).

Đổi `LOG_PATH` nếu cần; để trống/ sai thì tự tìm file `accuracy_log*.jsonl` mới nhất dưới `outputs/`.

In [16]:
import json, glob, os

# ==== Đường dẫn log (sửa ở đây nếu cần) ====
LOG_PATH = r"E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\outputs\Qwen_4B_Final_QA_result_test\accuracy_log (11).jsonl"
if not os.path.isfile(LOG_PATH):
    cands = glob.glob(os.path.join(os.getcwd(), '..', '**', 'accuracy_log*.jsonl'), recursive=True)
    if cands:
        LOG_PATH = max(cands, key=os.path.getmtime)
print('Đọc:', LOG_PATH)

rows = [json.loads(l) for l in open(LOG_PATH, encoding='utf-8') if l.strip()]
print(f'Số lần eval (epoch) trong log: {len(rows)}')
for i, r in enumerate(rows):
    print(f'  line {i}: accuracy={r["accuracy"]:.4f}  ({r["correct"]}/{r["total"]})')

# ==== Tự chọn epoch accuracy cao nhất ====
best_i = max(range(len(rows)), key=lambda i: rows[i]['accuracy'])
best = rows[best_i]
print(f'\n>>> CHỌN line {best_i}: accuracy={best["accuracy"]:.4f}  ({best["correct"]}/{best["total"]})  '
      f'avg_latency={best.get("avg_latency_sec", 0):.2f}s')

Đọc: e:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\notebooks\..\outputs\Qwen_4B_Final_QA_result_test\accuracy_log_11.jsonl
Số lần eval (epoch) trong log: 3
  line 0: accuracy=0.8200  (41/50)
  line 1: accuracy=0.8400  (42/50)
  line 2: accuracy=0.9400  (47/50)

>>> CHỌN line 2: accuracy=0.9400  (47/50)  avg_latency=14.02s


In [17]:
# ==== TẤT CẢ sample (gọn: idx / pred / gold / đúng-sai) ====
print('=' * 80)
print(f'BEST epoch  accuracy={best["accuracy"]:.4f}  correct={best["correct"]}/{best["total"]}')
print('=' * 80)
print(f'\n--- TẤT CẢ {len(best["samples"])} sample ---')
for s in best['samples']:
    mark = '✅' if s['correct'] else '❌'
    print(f'  {mark} idx={s["idx"]:<4} pred={str(s["pred_answer"]):<10} gold={str(s["gold_answer"]):<10}')
wrong = [s for s in best['samples'] if not s['correct']]
print(f'\n  SAI: {len(wrong)}/{len(best["samples"])} mẫu — idx: {[s["idx"] for s in wrong]}')

BEST epoch  accuracy=0.9400  correct=47/50

--- TẤT CẢ 50 sample ---
  ✅ idx=0    pred=Yes        gold=Yes       
  ✅ idx=1    pred=A          gold=A         
  ✅ idx=2    pred=Yes        gold=Yes       
  ✅ idx=3    pred=A          gold=A         
  ✅ idx=4    pred=Yes        gold=Yes       
  ✅ idx=5    pred=B          gold=B         
  ✅ idx=6    pred=Yes        gold=Yes       
  ✅ idx=7    pred=Yes        gold=Yes       
  ✅ idx=8    pred=Yes        gold=Yes       
  ✅ idx=9    pred=Yes        gold=Yes       
  ✅ idx=10   pred=B          gold=B         
  ✅ idx=11   pred=Yes        gold=Yes       
  ✅ idx=12   pred=B          gold=B         
  ✅ idx=13   pred=Yes        gold=Yes       
  ✅ idx=14   pred=A          gold=A         
  ✅ idx=15   pred=Yes        gold=Yes       
  ✅ idx=16   pred=A          gold=A         
  ✅ idx=17   pred=Yes        gold=Yes       
  ✅ idx=18   pred=A          gold=A         
  ✅ idx=19   pred=Yes        gold=Yes       
  ✅ idx=20   pred=A          go

In [18]:
# ==== FULL DETAIL: input + GOLD vs PREDICTION (premises_used + reasoning_steps + explanation) ====
# Lưu ý: full reasoning chỉ log cho các 'detail_samples' (mặc định 6 mẫu random/epoch).
def show_detail(d):
    inp = d['input']
    print('=' * 92)
    print(f'idx={d["idx"]}    {"✅ CORRECT" if d["correct"] else "❌ WRONG"}')
    print('=' * 92)
    if inp.get('premises_fol'):
        print('PREMISES (FOL):')
        for j, f in enumerate(inp['premises_fol']):
            print(f'  [{j}] {f}')
    print('\nQUESTION:\n  ' + str(inp.get('question', '')).replace('\n', '\n  '))
    for who in ('gold', 'prediction'):
        b = d.get(who, {})
        print(f'\n── {who.upper()} ──')
        print(f'  answer         : {b.get("answer")}')
        print(f'  premises_used  : {b.get("premises_used")}')
        print('  reasoning_steps:')
        for s in b.get('reasoning_steps', []) or []:
            print(f'      - {s}')
        print(f'  explanation    : {b.get("explanation", "")}')
    print()

print(f'{"#" * 92}\n#  FULL DETAIL — {len(best["detail_samples"])} mẫu (epoch accuracy cao nhất)\n{"#" * 92}\n')
for d in best['detail_samples']:
    show_detail(d)

############################################################################################
#  FULL DETAIL — 6 mẫu (epoch accuracy cao nhất)
############################################################################################

idx=1    ✅ CORRECT
PREMISES (FOL):
  [0] ∀x(¬Practice(x) → ¬Solve(x))
  [1] ∀x(Attend(x))
  [2] ∃x(Enrolled(x))
  [3] ∃x(Solve(x))
  [4] ∀x(AskQuestions(x) → Attend(x))
  [5] ∀x(¬Practice(x) → ¬AskQuestions(x))
  [6] ∀x(Solve(x) → Pass(x))
  [7] ∀x(Enrolled(x) → Motivated(x))
  [8] ∀x(Pass(x) → Graduate(x))
  [9] ∀x(Graduate(x) → Excel(x))

QUESTION:
  Based on the premises, which statement is correct?
  A. Students who solve math problems correctly must have practiced them.
  B. Some students ask questions in math class even if they do not attend lectures.
  C. All students who practice math problems ask questions in class.

── GOLD ──
  answer         : A
  premises_used  : [0]
  reasoning_steps:
      - Rule: ∀x(¬Practice(x) → ¬Solve(x))
      - Deriv